# Actor–Critic with GAE on LunarLander-v3

This notebook uses the exact same implementation as the command-line program. Change the configuration below, run training, and inspect the per-run logs, curve, checkpoints, and videos.

## Core equations

The critic supplies the temporal-difference residual $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$. GAE accumulates it backward as $A_t = \delta_t + \gamma\lambda A_{t+1}$. The actor increases the log-probability of actions with positive advantage; the critic regresses toward $A_t + V(s_t)$.

In [ ]:
from pathlib import Path
import csv
import sys

current = Path.cwd().resolve()
if current.name == 'notebooks':
    PROJECT_ROOT = current.parent
elif (current / 'src' / 'rl_actor_critic').exists():
    PROJECT_ROOT = current
elif (current / 'rl-actor-critic' / 'src').exists():
    PROJECT_ROOT = current / 'rl-actor-critic'
else:
    raise RuntimeError('Could not locate the rl-actor-critic project root')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from rl_actor_critic import TrainingConfig, train

print('Project root:', PROJECT_ROOT)

## Adjustable hyperparameters

Start with a short run to verify the pipeline. Increase `episodes` to 1000–2000 for a serious experiment. Episode 0 is the untrained policy; final evaluation is always recorded when `record_video=True`.

In [ ]:
config = TrainingConfig(
    episodes=20,
    gamma=0.99,
    gae_lambda=0.95,
    learning_rate=3e-4,
    hidden_size=128,
    value_loss_coef=0.5,
    entropy_coef=0.01,
    max_grad_norm=0.5,
    seed=20,
    evaluation_seed=2026,
    device='auto',
    log_interval=1,
    checkpoint_interval=10,
    record_episodes=(0, 1, 10),
    record_video=True,
)
config.validate()
config.to_dict()

In [ ]:
result = train(
    config,
    project_root=PROJECT_ROOT,
    run_name='notebook',
)
result

## Inspect the structured history

`metrics.csv` has one row per training iteration. Reading it with the standard library keeps the notebook independent of pandas.

In [ ]:
with (result.run_directory / 'metrics.csv').open() as handle:
    metrics = list(csv.DictReader(handle))

metrics[-5:]

In [ ]:
from IPython.display import Markdown, SVG, display

plot_files = sorted((result.run_directory / 'plots').rglob('*.svg'))
for plot_file in plot_files:
    plot_title = plot_file.stem.replace('_', ' ').title()
    display(Markdown(f'### {plot_title}'))
    display(SVG(filename=str(plot_file)))

## Watch checkpoint evaluations

Every video uses the same evaluation seed and greedy actions, making visual comparisons meaningful.

In [ ]:
from IPython.display import Video

video_files = sorted((result.run_directory / 'videos').glob('*.mp4'))
video_files

In [ ]:
# Change the index to compare episode 0, episode 1, milestones, and final.
Video(str(video_files[-1]), embed=True) if video_files else 'No video was recorded'